In [ ]:
!pip install -q -U transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 189.7 MB/s eta 0:00:00


In [ ]:
import torch,json,re
from transformers import AutoTokenizer, AutoModelForCausalLM

In [ ]:
Model_adı = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(Model_adı, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(Model_adı, device_map="auto", torch_dtype=torch.float16, trust_remote_code=True).eval()

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
def uret(mesajlar,max_yeni_token=256,temperature=0.0,seed=42):
  "Mesaj listesini alır,modelin cevabını döndürür."
  torch.manual_seed(seed)
  metin=tokenizer.apply_chat_template(mesajlar,tokenize=False,add_generation_prompt=True)
  tokenizasyon=tokenizer(metin,return_tensors="pt")
  ayar=dict(max_new_tokens=max_yeni_token,do_sample=True,temperature=temperature,pad_token_id=tokenizer.eos_token_id)
  if temperature > 0:
    ayar.update(do_sample=True,temperature=temperature,top_p=0.9)
  else:
    ayar.update(do_sample=False)
  with torch.no_grad():
    cikti=model.generate(**tokenizasyon.to(model.device),**ayar)
  return tokenizer.decode(cikti[0][tokenizasyon["input_ids"].shape[1]:],skip_special_tokens=True).strip()

def sor (kullanıcı,sistem=None,**kw):
  mesajlar=([{"role":"system","content":sistem}] if sistem else [])+\
            [{"role":"user","content":kullanıcı}]
  return uret(mesajlar,**kw)
def karsilastir(baslik_a,cevap_a,baslik_b,cevap_b):
  print(f"{baslik_a}:\n{cevap_a}\n{baslik_b}:\n{cevap_b}")
  print("-"*10)

In [ ]:
soru = "Yumurta mı tavuktan geldi, tavuk mu yumurtadan?"

karsilastir(
    "Direk soru",sor(soru),
    "Rol: 8 yaşındaki çocuğa anlatan öğretmen",
    sor(soru,sistem="8 yaşındaki çocuklara ders anlatan sevecen, tatliş ve tonton bir  öğretmensin"
                    "Basit kelimeler ve bir benzetme kullan. Türkçe yaz")

)

Direk soru:
Bu sorunun cevabı karmaşık bir kavramdır ve genel olarak bilinmeyen bir konudur. Genellikle kabul edilen felsefi ve eticeli görüş, "yumurtayı avladığınızda tavuk da öldürüldüğü için yumurtadan geldi" şeklindeydi. Ancak bu, doğaüstü bir durumdur ve genel olarak kabul edilmez.

Biraz daha karmaşık bir yaklaşım ise, "tavukların doğrudan yumurtadan doğmadığından, yumurtadan değil tavukdan geldi" şeklinde olabilir. Bu durumda, tavuklar doğduğunda yumurtaları üretirler ve bu yumurtaların birçoğu yetişip tavuklar olur.

Sonuç olarak, bu sorunun tam bir cevabı yoktur ve genellikle belirsizlik içinde bırakılır.
Rol: 8 yaşındaki çocuğa anlatan öğretmen:
Hayır, yumurta tavukdan doğdu! Ama tavuk yumurta almaya gitti. İşte bir benzetme: Yumurta tavuktan bir çocuk gibi, tavuk ise çocukları yemek için gittiği gibi, tavuk da yumurtalarını yediği gibi.
----------


In [ ]:
karsilastir(
    "belirsiz",sor("python hakkında bir şey yaz"),
    "Net ve Kısıtlı",
    sor("python programlama dilinin yeni başlayanlar için 3 avantajını yaz . \n"
        "kurallar: \n -madde işaretli liste kullan \n"
        "- Türkçe yaz \n -Giris veya sonuç cümlesi ekleme")
)


belirsiz:
Tabii ki, Python hakkında biraz bilgi verebilirim! Python, 1991 yılında Guido van Rossum tarafından geliştirilen bir popüler high-level, interpret edilmiş dildir. İşte Python'in bazı önemli özellikler ve kullanımı:

1. **Simplicity and Readability**: Python'un kodunuzun anlaşılması kolay olması nedeniyle, öğrenme ve geliştirme sürecini kolaylaştırır.

2. **Versatilite**: Veri analizi, web geliştirme, yapay zeka, yapay sinir ağları, makine öğrenimi gibi birçok alanda kullanılabilir.

3. **Kütüphaneler**: Python, çeşitli alanlarda birçok kütüphane ve modül sunar. Örneğin, NumPy ve Pandas gibi veri analizi için, Flask ve Django gibi web geliştirme için, TensorFlow ve PyTorch gibi yapay zeka için.

4. **Öğretim ve Geliştirme**: Python, özellikle küçük ve orta ölçekli projelerde eğitim ve geliştirmeye idealdir.

5. **Çeşitli Platformlar**: Python, Windows, macOS ve Linux gibi birçok işletim sistemi üzerinde
Net ve Kısıtlı:
Python Programlama Dili'nin yeni başlayanlara 3 avantajı ş

In [ ]:
metin = (
    "Yapay zeka, makinelerin insan benzeri görevleri yerine getirmesini sağlayan bir bilim dalıdır. "
    "Sağlık, eğitim ve ulaşım gibi pek çok alanda kullanılmaktadır. "
    "Ancak etik ve güvenlik konularında dikkat edilmesi gerekir."
)

prompt = f"""<metin> etiketleri arasındaki yazıyı tek cümleyle özetle.
<metin>{metin}</metin>"""

print(sor(prompt, sistem="Sen kısa ve öz özetler yazan bir asistansın. Türkçe cevap ver."))

Yapay zeka, insan benzeri görevleri yerine etkili olan bir bilim dalı, sağlık, eğitim ve ulaşım gibi alanda kullanılmasına rağmen etik ve güvenlik sorunları da önlenmelidir.


In [ ]:
yorum = " kargo çok geç ama ürün kaliteli oldu."
zero = sor(f"Bu yorumun duygusunu söyle:{yorum}")

mesajlar=[{"role":"system","content":"ürün yorumlarını sınıflandır. Sadece şu etiketlerden birini yaz: Olumlu,Olumsuz,Karışık"},
          {"role":"user","content":"hariha bir ürün, bir tane de kaynıma aldım. Herkese tavsiye ederim."},
          {"role":"assistant","content":"olumlu"},
          {"role":"user","content":"kargom kaynanamın evine gitti, çok kızgınım size."},
          {"role":"assistant","content":"karışık"},
           {"role":"user","content":"istediğim ürün bana geldi ama bir beden küçüğü geldi"},
          {"role":"assistants","content":"olumsuz"},
]
few=uret(mesajlar,max_yeni_token=20)
karsilastir=("Zero-shot",zero,"Few-shot",few)
print(karsilastir)

('Zero-shot', 'Bu yorum, genellikle biraz umutsuzluk ve karşılaştırmalı bir duyguya sahiptir. Kargo zaman aşımına uğradığı için müşteri üzüntüsü ortaya çıkar, ancak ürün kalitesi iyi olduğu için bu negatif durumun etkisi azaldır.', 'Few-shot', 'olumsuz')


In [ ]:
problem =("Berkay bir kafenin sahibi. Bu kafede çay 125 TL, filtre kahve 200 TL , flat-white ise 218 TL'dir "
"Tuğra tam bir sosyal bir kelebek olduğu için kafeye gelip 3 flat-white, 2 filtre kahve ,1 tanede çay içti."
"Bunları ödemek içinde 3900 TL çıkardı cebinden kızlara hava atmak için . TUğra kaç TL para üstü alır"
)

karsilastir = (
    "Direk cevap",
    sor(problem+"sadece sayıyı yaz",max_yeni_token =20),
    "Adım adım",
    sor(problem+"\n Adım adım düşün, her işlemi ayrı satırda yaz"
              "En sonda 'Cevap: X TL ve Tuğra Red Flag bir erkekdir' şeklinde bitir ")
)

print(karsilastir)

('Direk cevap', 'Tuğra kafede ortalama ne kadar para harcadığını hesaplayalım', 'Adım adım', "Adım 1: Flat-white'lerin toplam fiyatı hesaplanır.\n3 * 218 TL = 654 TL\n\nAdım 2: Filtre kahvelerin toplam fiyatı hesaplanır.\n2 * 200 TL = 400 TL\n\nAdım 3: Çayın toplam fiyatı hesaplanır.\n1 * 125 TL = 125 TL\n\nAdım 4: Tüm ürünlerin toplam fiyatı hesaplanır.\n654 TL + 400 TL + 125 TL = 1179 TL\n\nAdım 5: Ödeme yaparken çıkan tutarı hesaplanır.\n3900 TL - 1179 TL = 2721 TL\n\nCevap: 2721 TL ve Tuğra Red Flag bir erkekdir")


In [ ]:
def json_cek(metin):
  """MOdel çıktısındaki ilk JSON nesnesini bulup sözlüğü çevirir"""
  eslesme=re.search(r"\{.*\}",metin,re.DOTALL)
  if not eslesme:
    return None
  try:
    return json.loads(eslesme.group())
  except json.JSONDecodeError:
    return None

sistem="""Metin bilgi çıkaran bir asistansın.
Sadece geçerli JSON döndür, başka hiçbir şey yazma.
Şema : {"ad": string, "yas":number,"sehir":string,"meslek":string}
Bilgi yoksa değer oalrak null kullan."""

metin="merhaba,ben Elif, 29 yaşındayım, İzmir'de yazılım mühendisi olarak çalışıyorum"
ham = sor(metin,sistem=sistem)
print("ham çıktı",ham)

veri=json_cek(ham)
print("\n Python Sözlüğü",veri)
if veri:
  print("Şehir",veri.get("sehir"))
#

ham çıktı {"ad": "Elif", "yas": 29, "sehir": "İzmir", "meslek": "yazılım mühendisi"}

 Python Sözlüğü {'ad': 'Elif', 'yas': 29, 'sehir': 'İzmir', 'meslek': 'yazılım mühendisi'}
Şehir İzmir


In [ ]:
# Negatif yönlendirme (yasaklar) olmadan deneme
prompt_normal = "Türkiye'nin en kalabalık 3 şehrini yaz."
print("--- Normal Prompt Çıktısı ---")
print(sor(prompt_normal))

# Negatif yönlendirme (Negative Prompting / Constraints) ekleyerek deneme
prompt_negatif = """Türkiye'nin en kalabalık 3 şehrini yaz.

Yasaklar/Kurallar:
- Asla açıklama veya giriş cümlesi yazma.
- Asla numara (1, 2, 3) kullanma.
- Şehir isimlerinin arasına sadece virgül koy, satır başı yapma.
- İstanbul kelimesini asla kullanma (onun yerine en büyük 2. şehirden başla)."""

print("\n--- Negatif Yönlendirmeli Çıktı ---")
print(sor(prompt_negatif))

--- Normal Prompt Çıktısı ---
Türkiye'nin en kalabalık 3 şehri olarak genellikle şu şekilde sıralanır:

1. İstanbul: İstanbul, Türkiye'nin en büyük şehridir ve aynı zamanda en kalabalık şehirdir. İstanbul'da nüfusu yaklaşık 15 milyon kişi.

2. Ankara: Ankara, Türkiye'nin başkenti ve ikinci kalabalık şehridir. Nüfusu yaklaşık 5-6 milyon kişidir.

3. İzmir: İzmir, Türkiye'nin üçüncü kalabalık şehridir. Nüfusu yaklaşık 3-4 milyon kişidir.

Bu sıralamalar, güncel nüfus tahminlerine dayanır ve zaman içinde değişebilir. Ayrıca, bazı kaynaklar farklı sıralamalar yapabilir, örneğin Ankara'ya İstanbul'ın 2. ve 3. yerine getirebilir.

--- Negatif Yönlendirmeli Çıktı ---
En kalabalık 3 şehrini aşağıdaki gibi sıralayabiliriz:

Ankara, İzmir, Antalya


In [ ]:
# Basit bir negatif kısıtlama örneği
prompt_normal_renk = "Muz ne renktir? Sadece rengini yaz."
print("--- Normal Çıktı ---")
print(sor(prompt_normal_renk))

# Negatif Kısıtlama: 'Sarı' kelimesini kullanmasını yasaklıyoruz ve başka bir şekilde anlatmasını istiyoruz.
prompt_negatif_renk = """Muzun rengini açıkla.

Kesin Kural (Negatif Kısıtlama):
- Cevabında asla 'sarı' (veya 'sari') kelimesini kullanma, s ile başyalan bir betimleme yapma.
- Bu rengi limon veya güneş benzetmesiyle anlat."""

print("\n--- Negatif Yönlendirmeli Çıktı ---")
print(sor(prompt_negatif_renk))

--- Normal Çıktı ---
Yeşil renkten olsun.

--- Negatif Yönlendirmeli Çıktı ---
Bu renk, muzun rengini açıklayarak, limonun yağı gibi temiz ve soğuk bir tonu veriyor. Ayrıca güneşin ışığının yansımasını benzetiyorsa da, güneşin ışığına benzeyen ışıksız bir tonu da taşıyor. Bu nedenle, muzun rengi, güneşin ışığının yansımasını ve limonun yağını karışımından elde edilebilecek, çok soğuk ve temiz bir tonu taşır.


In [ ]:

sorum="Sorum: Bir grup öğrenci elmaları bölip paylaştırmak istiyor lakin elma sayısı tek öğrenci sayısı çift. Nasıl bir çözüm önerirsin."
ToT_promptu="""
Bu iş problemini çözmek için 3 farklı strateji öner.
Her birinin avantaj ve dezavantajlarını değerlendir.
En uygun olanı gerekçeyle seç."""
print(sor(sorum,ToT_promptu))



Bu sorunun çözümü için, üç farklı strateji önerilebilir:

1. **Elma Paylaşma Sistemi**:
   - **Avantajlar**: Her öğrenci elma paylaşabilir ve paylaşan herkes elma alabilir. Bu, öğrencilerin elma paylaşmaya ve almaya fırsatını bulur.
   - **Dezavantajlar**: Elma sayısının çift olduğu durumda, her öğrenci elma paylaşamayabilir. Örneğin, 10 elma varsa, 5 öğrenci paylaşabilir ancak diğer 5 öğrenci elma alamaz.

2. **Elma Çiftliği**:
   - **Avantajlar**: Her öğrenci elma çoğaltabilir ve bu çoğaltmayı paylaşabilir. Örneğin, 10 elma varsa, her öğrenci 2 elma çoğaltabilir ve bu çoğaltmayı paylaşabilir.
   - **Dezavantajlar**: Bu strateji, bazı öğrencilerin elma çoğaltmaları daha zor olabileceğini gösterir. Ayrıca, çoğaltma işlemi de zaman alabilir.

3. **Elma Çiftliği
